<a href="https://colab.research.google.com/github/n2pakbong/boundary-adapted-pinns/blob/main/experiments/mean_escape_time/ou_ball/ou_mean_escape_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ornstein–Uhlenbeck mean escape time: five-method comparison

This is the principal notebook for the OU experiment in the paper. It keeps the original problem, five neural formulations, Monte Carlo data term, adaptive loss weights, optimizer schedule, and error measures in one readable run.

**Suggested path:** read Sections 1–3, choose `RUN_SCALE`, prepare the shared data once, train the selected methods, then compare all results in the same figures and tables. `quick` is a workflow demonstration. `paper` restores the recorded experiment counts and is expensive.


## Notebook route

Read from top to bottom: define the OU problem, choose `quick` or `paper`, inspect the exact reference, generate one shared Monte Carlo dataset, train the selected methods, and compare them on the same grid. Long stages report progress and save checkpoints that can be resumed.


## 1. Problem and reference solution

In the disk $\Omega=B_R(0)\subset\mathbb R^2$, consider

$$dX_t=-\theta X_t\,dt+\sigma\,dB_t,$$

and the first exit time $T=\inf\{t\geq0:X_t\in\partial\Omega\}$. Its mean $\tau(x)=\mathbb E_x[T]$ solves

$$-\mathcal L\tau=1\quad\text{in }\Omega,\qquad \tau=0\quad\text{on }\partial\Omega,$$

with $\mathcal L u=-\theta x\cdot\nabla u+\frac{\sigma^2}{2}\Delta u$.

The recorded experiment uses $R=2$, $\theta=3$, and **$\sigma^2=2$**. For $d$ dimensions, Kersting et al. give

$$
\tau(x)=\frac{1}{\lambda^{d/2}\sigma^2}
\int_{\lVert x\rVert}^{R}z^{1-d}e^{\lambda z^2}
\gamma\!\left(\frac d2,\lambda z^2\right)\,dz,
\qquad \lambda=\frac{\theta}{\sigma^2},
$$

where $\gamma(s,y)=\int_0^y t^{s-1}e^{-t}\,dt$. In two dimensions this becomes

$$
\tau(x)=\frac1\theta\int_{\lVert x\rVert}^{R}
\frac{e^{\lambda z^2}-1}{z}\,dz.
$$

The notebook evaluates this integral for the reference field. It also generates shared Euler–Maruyama labels because that data term was part of every original training objective.


## 2. The five methods

All networks have three hidden layers of width 30, `tanh` activations, Xavier-normal weights with gain 2, and double precision.

| Name in code | Trial function | Loss components |
|---|---|---|
| `standard` | $N_\theta$ | strong PDE, sampled boundary, Monte Carlo data |
| `adapted` | $dN_\theta$ | strong PDE, Monte Carlo data |
| `over` | $d^{3/2}N_\theta$ | strong PDE, Monte Carlo data |
| `under` | $d^{1/2}N_\theta$ | strong PDE, Monte Carlo data |
| `variational` | $N_\theta$ | weak residual, Nitsche boundary term, Monte Carlo data |

Here $d(x)=(R^2-\lVert x\rVert^2)/(2R)$ vanishes on the circle. The three exact-boundary methods use

$$u_\theta(x)=d(x)^\alpha N_\theta(x),\qquad
\alpha\in\{1,3/2,1/2\}.$$

The strong residual and Monte Carlo data losses are

$$J_{\rm pde}=\frac1{N_\Omega}\sum_i|\mathcal L u_\theta(x_i)+1|^2,
\qquad
J_{\rm data}=\frac1{N_d}\sum_j|u_\theta(z_j)-\widehat\tau_j|^2.$$

The standard PINN also uses $J_{\rm bc}=N_{\partial\Omega}^{-1}\sum_j|u_\theta(y_j)|^2$. Each active component $J_i$ is combined exactly as in the original notebooks,

$$J_{\rm train}=\sum_i e^{-2s_i}J_i+\sum_i s_i,$$

with trainable log scales $s_i$. **Checkpoint correction:** the original variational saved-model score counted its weak loss twice and omitted boundary and data; the other original selectors also omitted trained components. The corrected score uses the same weighted objective as training. The training objectives themselves are unchanged.


In [ ]:
# Colab setup: this notebook is self-contained and needs no repository token.
import math
import time
from collections import OrderedDict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import integrate
import torch
from torch import nn

torch.set_default_dtype(torch.float64)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", DEVICE)


## 3. Editable configuration

The `paper` values below are the settings recorded in the original notebooks. The quick preset changes only sample counts and iteration budgets; it does not change the equation, method, architecture, or loss definitions.


In [ ]:
RUN_SCALE = "quick"                 # "quick" or "paper"
METHODS = ["standard", "adapted", "over", "under", "variational"]
RUN_TRAINING = False                 # set True when ready
RUN_LBFGS = False                    # paper workflow used Adam then L-BFGS
RESUME = True
SEED = 2026

R = 2.0
theta = 3.0
sigma_squared = 2.0
sigma = math.sqrt(sigma_squared)

width = 30
depth = 3
initialization_gain = 2.0

if RUN_SCALE == "paper":
    pool_size, batch_size, adam_epochs = 8192, 512, 10_000
    data_anchors, paths_per_anchor = 256, 256
    mc_dt, mc_max_time = 5e-4, 30.0
    lbfgs_iterations = adam_epochs // 2
    report_every, checkpoint_every = 100, 500
elif RUN_SCALE == "quick":
    pool_size, batch_size, adam_epochs = 512, 128, 40
    data_anchors, paths_per_anchor = 24, 24
    mc_dt, mc_max_time = 5e-3, 2.0
    lbfgs_iterations = 20
    report_every, checkpoint_every = 10, 10
else:
    raise ValueError("RUN_SCALE must be 'quick' or 'paper'")

learning_rate = 5e-4
number_of_decays = 100
final_learning_rate_factor = 1e-3
output_root = Path("/content/drive/MyDrive/boundary-adapted-pinns-runs/ou") if Path("/content/drive/MyDrive").is_dir() else Path("runs/ou")
output_root.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
print({"scale": RUN_SCALE, "output": str(output_root), "sigma_squared": sigma_squared})


### Reference solution

For dimension two, the lower incomplete-gamma expression simplifies to an integral with integrand $(e^{\lambda z^2}-1)/z$, where $\lambda=\theta/\sigma^2$. The special value at zero is taken by continuity. This is the same reference formula used by the original notebooks, evaluated more stably near the origin.


In [ ]:
def true_tau_radius(radius):
    lam = theta / sigma_squared

    def integrand(z):
        return 0.0 if z == 0.0 else np.expm1(lam * z * z) / z

    value, _ = integrate.quad(integrand, float(radius), R, epsabs=1e-10, epsrel=1e-10)
    return value / theta


def true_tau(points):
    radii = np.linalg.norm(np.asarray(points), axis=1)
    return np.array([true_tau_radius(r) for r in radii])


grid_n = 200
gx, gy = np.meshgrid(np.linspace(-R, R, grid_n), np.linspace(-R, R, grid_n))
disk_mask = gx**2 + gy**2 <= R**2
grid_points = np.column_stack((gx[disk_mask], gy[disk_mask]))
reference_grid = np.zeros_like(gx)
reference_grid[disk_mask] = true_tau(grid_points)

plt.figure(figsize=(5, 4))
plt.contourf(gx, gy, reference_grid, levels=50)
plt.colorbar(label=r"$\tau(x)$")
plt.gca().set_aspect("equal")
plt.title("OU reference mean escape time")
plt.show()


## 4. Shared sampling and Monte Carlo labels

Points are uniform by area in the disk. From an anchor $z$, Euler–Maruyama advances

$$X_{n+1}=X_n-\theta X_n\,\Delta t+\sigma\sqrt{\Delta t}\,\xi_n,
\qquad \xi_n\sim\mathcal N(0,I_2),$$

until the path leaves the disk. The label $\widehat\tau(z)$ is the mean of the simulated exit times. Each label averages 256 paths in the paper setting. A path still inside at `mc_max_time` is recorded at that cap, matching the original implementation; the progress report states how many paths remain capped.


In [ ]:
def sample_disk(count, *, device=DEVICE):
    angle = 2 * math.pi * torch.rand(count, device=device)
    radius = R * torch.sqrt(torch.rand(count, device=device))
    return torch.stack((radius * torch.cos(angle), radius * torch.sin(angle)), dim=1)


def sample_boundary(count, *, device=DEVICE):
    angle = 2 * math.pi * torch.rand(count, device=device)
    return torch.stack((R * torch.cos(angle), R * torch.sin(angle)), dim=1)


@torch.no_grad()
def monte_carlo_exit_time(x0, n_paths, dt, max_time):
    x0 = x0.to(DEVICE)
    batch = x0.shape[0]
    positions = x0[:, None, :].expand(batch, n_paths, 2).clone()
    elapsed = torch.zeros(batch, n_paths, device=DEVICE)
    alive = torch.ones(batch, n_paths, dtype=torch.bool, device=DEVICE)
    max_steps = int(max_time / dt)
    interval = max(1, max_steps // 100)
    noise_scale = sigma * math.sqrt(dt)
    started = time.time()

    for step in range(max_steps):
        if not alive.any():
            break
        current = positions[alive]
        positions[alive] = current - theta * current * dt + noise_scale * torch.randn_like(current)
        elapsed[alive] += dt
        active_indices = alive.nonzero(as_tuple=False)
        exited = positions[alive].square().sum(dim=1) >= R**2
        if exited.any():
            ids = active_indices[exited]
            alive[ids[:, 0], ids[:, 1]] = False
        if step % interval == 0 or step + 1 == max_steps:
            print(f"MC step {step + 1:6d}/{max_steps} | alive {alive.float().mean():6.2%} | {time.time() - started:6.1f}s", flush=True)

    if alive.any():
        print(f"Censored at {max_time:g}: {int(alive.sum())}/{alive.numel()} paths")
    return elapsed.mean(dim=1)


## 5. Editable models and differential operators

`OUModel` holds the common MLP. Only the trial function and active adaptive weights change between methods. Editing `width`, `depth`, the activation, or initialization changes every selected method consistently.


In [ ]:
METHOD_SPECS = OrderedDict([
    ("standard", {"power": None, "losses": ("pde", "data", "bc")}),
    ("adapted", {"power": 1.0, "losses": ("pde", "data")}),
    ("over", {"power": 1.5, "losses": ("pde", "data")}),
    ("under", {"power": 0.5, "losses": ("pde", "data")}),
    ("variational", {"power": None, "losses": ("var", "data", "bc")}),
])


def smooth_distance(x):
    return (R**2 - x.square().sum(dim=1)) / (2 * R)


class OUModel(nn.Module):
    def __init__(self, method):
        super().__init__()
        self.method = method
        self.power = METHOD_SPECS[method]["power"]
        layers = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers.extend((nn.Linear(width, width), nn.Tanh()))
        layers.append(nn.Linear(width, 1))
        self.network = nn.Sequential(*layers)
        initial = {name: -0.5 for name in METHOD_SPECS[method]["losses"]}
        if method == "variational":
            initial["bc"] = 3.0  # recorded OU Nitsche initialization
        self.log_scales = nn.ParameterDict({name: nn.Parameter(torch.tensor(value)) for name, value in initial.items()})
        self.apply(self._initialize)

    @staticmethod
    def _initialize(module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_normal_(module.weight, gain=initialization_gain)
            nn.init.constant_(module.bias, initialization_gain)

    def forward(self, x):
        raw = self.network(x)
        if self.power is None:
            return raw
        return raw * smooth_distance(x).clamp_min(0).pow(self.power).unsqueeze(1)


def strong_residual(model, points):
    x = points.detach().clone().requires_grad_(True)
    tau = model(x).squeeze(1)
    grad = torch.autograd.grad(tau.sum(), x, create_graph=True)[0]
    tau_xx = torch.autograd.grad(grad[:, 0].sum(), x, create_graph=True)[0][:, 0]
    tau_yy = torch.autograd.grad(grad[:, 1].sum(), x, create_graph=True)[0][:, 1]
    generator = -theta * (x * grad).sum(dim=1) + 0.5 * sigma_squared * (tau_xx + tau_yy)
    return generator + 1.0


### Variational/Nitsche residual

Write $A=\sigma^2/2$. Multiplying $-\mathcal Lu=1$ by a test function $\phi$ and integrating the diffusion term by parts gives

$$
A\int_\Omega\nabla u\cdot\nabla\phi\,dx
+\theta\int_\Omega(x\cdot\nabla u)\phi\,dx
-A\int_{\partial\Omega}(\nabla u\cdot n)\phi\,ds
=\int_\Omega\phi\,dx.
$$

The symmetric Nitsche term is $-A\int_{\partial\Omega}u\,\nabla\phi\cdot n\,ds$; the raw penalty $\int_{\partial\Omega}u\phi\,ds$ is the separately weighted boundary component. The notebook uses 50 Gaussian tests

$$\phi_k(x)=\exp\!\left(-\frac{|x-c_k|^2}{2s^2}\right),\qquad s=R/5,$$

and estimates the integrals by sample averages.


In [ ]:
n_test_functions = 50
test_length = R / 5.0
test_centers = sample_disk(n_test_functions)


def variational_residual(model, interior, boundary):
    interior = interior.detach().clone().requires_grad_(True)
    boundary = boundary.detach().clone().requires_grad_(True)
    u_int = model(interior).squeeze(1)
    grad_u = torch.autograd.grad(u_int.sum(), interior, create_graph=True)[0]

    delta = interior[:, None, :] - test_centers[None, :, :]
    phi = torch.exp(-delta.square().sum(dim=2) / (2 * test_length**2))
    grad_phi = -delta * phi[..., None] / test_length**2
    area = math.pi * R**2
    diffusion = 0.5 * sigma_squared * (grad_u[:, None, :] * grad_phi).sum(dim=2)
    drift = theta * (interior * grad_u).sum(dim=1, keepdim=True) * phi
    weak = (diffusion + drift).mean(dim=0) * area - phi.mean(dim=0) * area

    delta_b = boundary[:, None, :] - test_centers[None, :, :]
    phi_b = torch.exp(-delta_b.square().sum(dim=2) / (2 * test_length**2))
    grad_phi_b = -delta_b * phi_b[..., None] / test_length**2
    normal = boundary / R
    u_b = model(boundary).squeeze(1)
    grad_u_b = torch.autograd.grad(u_b.sum(), boundary, create_graph=True)[0]
    arclength = 2 * math.pi * R
    flux_u = (grad_u_b * normal).sum(dim=1)
    flux_phi = (grad_phi_b * normal[:, None, :]).sum(dim=2)
    weak = weak - 0.5 * sigma_squared * (flux_u[:, None] * phi_b).mean(dim=0) * arclength
    symmetry = -0.5 * sigma_squared * (u_b[:, None] * flux_phi).mean(dim=0) * arclength
    raw_boundary = (u_b[:, None] * phi_b).mean(dim=0) * arclength
    return weak + symmetry, raw_boundary


def adaptive_objective(model, components):
    return sum(torch.exp(-2 * model.log_scales[name]) * value + model.log_scales[name] for name, value in components.items())


def component_losses(model, method, interior, boundary, data_x, data_y):
    data_loss = (model(data_x).squeeze(1) - data_y).square().mean()
    if method == "variational":
        weak, raw_boundary = variational_residual(model, interior, boundary)
        return {"var": weak.square().mean(), "data": data_loss, "bc": raw_boundary.square().mean()}
    losses = {"pde": strong_residual(model, interior).square().mean(), "data": data_loss}
    if method == "standard":
        losses["bc"] = model(boundary).square().mean()
    return losses


## 6. Prepare shared data

This cell creates one Monte Carlo dataset and one fixed checkpoint-evaluation set for all five methods. It saves the labels so an interrupted training session does not need to simulate them again.


In [ ]:
data_path = output_root / f"ou_{RUN_SCALE}_shared_data.pt"
if data_path.exists() and RESUME:
    shared = torch.load(data_path, map_location=DEVICE)
    x_data, tau_data = shared["x"].to(DEVICE), shared["tau"].to(DEVICE)
    print("Loaded shared labels:", data_path)
else:
    x_data = sample_disk(data_anchors)
    tau_data = monte_carlo_exit_time(x_data, paths_per_anchor, mc_dt, mc_max_time)
    torch.save({"x": x_data.cpu(), "tau": tau_data.cpu(), "settings": {"dt": mc_dt, "max_time": mc_max_time, "paths": paths_per_anchor}}, data_path)
    print("Saved shared labels:", data_path)

evaluation_interior = sample_disk(pool_size)
evaluation_boundary = sample_boundary(max(1, pool_size // 4))
print(f"Shared labels: {len(x_data)} anchors × {paths_per_anchor} paths")


## 7. Adam, checkpointing, and optional L-BFGS

Each method writes `last.pt` for resumption, `best_adam.pt` for the lowest corrected weighted validation objective, and `final.pt` after optional L-BFGS. The checkpoint objective contains every loss component used to train that method. Historical displayed results below predate this correction.


In [ ]:
def make_scheduler(optimizer):
    step = max(1, adam_epochs // number_of_decays)
    gamma = math.exp(math.log(final_learning_rate_factor) / number_of_decays)
    return torch.optim.lr_scheduler.StepLR(optimizer, step_size=step, gamma=gamma)


def detached_row(components):
    return {name: float(value.detach()) for name, value in components.items()}


def train_method(method):
    run_dir = output_root / f"{RUN_SCALE}_{method}"
    run_dir.mkdir(parents=True, exist_ok=True)
    model = OUModel(method).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = make_scheduler(optimizer)
    history, start_epoch, best_score = [], 0, math.inf
    last_path, best_path = run_dir / "last.pt", run_dir / "best_adam.pt"

    if RESUME and last_path.exists():
        state = torch.load(last_path, map_location=DEVICE)
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        history, start_epoch, best_score = state["history"], state["epoch"], state["best_score"]
        print(f"{method}: resuming after epoch {start_epoch}")

    started = time.time()
    for epoch in range(start_epoch + 1, adam_epochs + 1):
        model.train()
        optimizer.zero_grad()
        components = component_losses(model, method, sample_disk(batch_size), sample_boundary(batch_size), x_data, tau_data)
        objective = adaptive_objective(model, components)
        objective.backward()
        optimizer.step()
        scheduler.step()

        model.eval()
        validation = component_losses(model, method, evaluation_interior, evaluation_boundary, x_data, tau_data)
        score = float(adaptive_objective(model, validation).detach())
        row = {"epoch": epoch, "objective": float(objective.detach()), "checkpoint_objective": score, **detached_row(components)}
        history.append(row)
        if score < best_score:
            best_score = score
            torch.save({"model": model.state_dict(), "epoch": epoch, "score": score}, best_path)

        if epoch % report_every == 0 or epoch == 1 or epoch == adam_epochs:
            pieces = " | ".join(f"{key} {value:.3e}" for key, value in detached_row(components).items())
            print(f"{method:11s} epoch {epoch:5d}/{adam_epochs} | {pieces} | {time.time() - started:.1f}s", flush=True)
        if epoch % checkpoint_every == 0 or epoch == adam_epochs:
            torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "history": history, "epoch": epoch, "best_score": best_score}, last_path)

    model.load_state_dict(torch.load(best_path, map_location=DEVICE)["model"])
    if RUN_LBFGS:
        fixed_interior = sample_disk(pool_size)
        fixed_boundary = sample_boundary(pool_size)
        tolerance = 1e-9 if method in {"standard", "variational"} else 1e-10
        lbfgs = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=lbfgs_iterations, history_size=1000, tolerance_grad=tolerance, tolerance_change=tolerance, line_search_fn="strong_wolfe")
        calls = 0

        def closure():
            nonlocal calls
            calls += 1
            lbfgs.zero_grad()
            parts = component_losses(model, method, fixed_interior, fixed_boundary, x_data, tau_data)
            loss = adaptive_objective(model, parts)
            loss.backward()
            if calls % 100 == 0:
                print(method, "L-BFGS call", calls, detached_row(parts), flush=True)
            return loss

        lbfgs.step(closure)
    torch.save({"model": model.state_dict(), "history": history, "method": method, "scale": RUN_SCALE}, run_dir / "final.pt")
    return model, pd.DataFrame(history)


In [ ]:
trained = {}
histories = {}
if RUN_TRAINING:
    for method in METHODS:
        if method not in METHOD_SPECS:
            raise ValueError(f"Unknown method: {method}")
        trained[method], histories[method] = train_method(method)
    print("Training complete:", ", ".join(trained))
else:
    print("Set RUN_TRAINING = True in Section 3 when you are ready.")


## 8. Evaluation

The following cells reproduce the original empirical relative $L^1$, $L^2$, $H^1$, and $H^2$ calculations on a Cartesian grid. The derivative norms use finite differences on the zero-filled disk grid, matching the historical evaluation convention.


In [ ]:
def evaluate_model(model):
    model.eval()
    with torch.no_grad():
        prediction = model(torch.as_tensor(grid_points, device=DEVICE)).squeeze(1).cpu().numpy()
    predicted_grid = np.zeros_like(gx)
    predicted_grid[disk_mask] = prediction
    truth = reference_grid[disk_mask]
    spacing = 2 * R / (grid_n - 1)
    error_grid = predicted_grid - reference_grid
    error_y, error_x = np.gradient(error_grid, spacing, spacing)
    error_xx = np.gradient(error_x, spacing, axis=1)
    error_yy = np.gradient(error_y, spacing, axis=0)
    error_xy = np.gradient(error_y, spacing, axis=1)
    truth_y, truth_x = np.gradient(reference_grid, spacing, spacing)
    truth_xx = np.gradient(truth_x, spacing, axis=1)
    truth_yy = np.gradient(truth_y, spacing, axis=0)
    truth_xy = np.gradient(truth_y, spacing, axis=1)
    rel_l1 = np.mean(np.abs(prediction - truth)) / np.mean(np.abs(truth))
    rel_l2 = np.sqrt(np.mean((prediction - truth)**2) / np.mean(truth**2))
    rel_h1 = np.sqrt(np.mean(error_grid[disk_mask]**2 + error_x[disk_mask]**2 + error_y[disk_mask]**2) / np.mean(reference_grid[disk_mask]**2 + truth_x[disk_mask]**2 + truth_y[disk_mask]**2))
    rel_h2 = np.sqrt(np.mean(error_grid[disk_mask]**2 + error_x[disk_mask]**2 + error_y[disk_mask]**2 + error_xx[disk_mask]**2 + error_yy[disk_mask]**2 + 2 * error_xy[disk_mask]**2) / np.mean(reference_grid[disk_mask]**2 + truth_x[disk_mask]**2 + truth_y[disk_mask]**2 + truth_xx[disk_mask]**2 + truth_yy[disk_mask]**2 + 2 * truth_xy[disk_mask]**2))
    return predicted_grid, {"L1 relative": rel_l1, "L2 relative": rel_l2, "H1 relative": rel_h1, "H2 relative": rel_h2}


if trained:
    figures = {"reference": reference_grid}
    metrics = {}
    for name, model in trained.items():
        figures[name], metrics[name] = evaluate_model(model)
    display(pd.DataFrame(metrics).T)
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
    for axis, (name, values) in zip(axes.flat, figures.items()):
        plot = axis.contourf(gx, gy, values, levels=50)
        axis.set_title(name)
        axis.set_aspect("equal")
        fig.colorbar(plot, ax=axis)
    for axis in axes.flat[len(figures):]:
        axis.axis("off")
    plt.show()


## 9. Historical paper-run summary

These are the means and sample standard deviations retained in the original result notebook. The raw run logs are absent, so the table cannot be regenerated from files in this repository. It predates the corrected Adam checkpoint selection and the corrected variational tensor contractions. Treat it as provenance for the paper comparison, not output from this reorganized notebook.

| Method | Relative L² | Relative H¹ | Relative H² |
|---|---:|---:|---:|
| Boundary-adapted | 2.006e-3 ± 3.014e-4 | 1.998e-3 ± 2.981e-4 | 1.990e-3 ± 2.946e-4 |
| Standard | 4.887e-2 ± 3.872e-2 | 4.994e-2 ± 3.926e-2 | 5.041e-2 ± 3.964e-2 |
| Variational/Nitsche | 1.169e-1 ± 5.697e-2 | 6.491e-1 ± 6.106e-1 | 3.269 ± 4.211 |
| Over-enforced $d^{3/2}$ | 1.013 ± 5.774e-4 | 1.006 ± 0 | 1.001 ± 5.774e-4 |
| Under-enforced $d^{1/2}$ | 1.009 ± 1.732e-3 | 1.005 ± 5.774e-4 | 1.000 ± 0 |


## References

- N. Tepakbong, J. Fan, X. Zhou, and D.-X. Zhou, [“Boundary-Adapted PINNs for Elliptic Dirichlet Problems”](https://arxiv.org/abs/2607.19167), 2026.
- H. Kersting, A. Orvieto, F. Proske, and A. Lucchi, [“Mean first exit times of Ornstein–Uhlenbeck processes in high-dimensional spaces”](https://arxiv.org/abs/2208.04029), *J. Phys. A* 56 (2023), 215003.
- M. Raissi, P. Perdikaris, and G. E. Karniadakis, [“Physics-informed neural networks”](https://doi.org/10.1016/j.jcp.2018.10.045), *JCP* 378 (2019).
- N. Sukumar and A. Srivastava, [“Exact imposition of boundary conditions with distance functions”](https://doi.org/10.1016/j.cma.2021.114333), *CMAME* 389 (2022).
- Y. Liao and P. Ming, [“Deep Nitsche Method”](https://doi.org/10.4208/cicp.OA-2020-0219), *Commun. Comput. Phys.* 29 (2021).
- A. Kendall, Y. Gal, and R. Cipolla, [“Multi-task learning using uncertainty to weigh losses”](https://openaccess.thecvf.com/content_cvpr_2018/html/Kendall_Multi-Task_Learning_Using_CVPR_2018_paper.html), CVPR 2018.
